In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime

class DailyTickerDownloader:
    """Download daily data for tickers with date-based folder organization"""
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = 'https://www.alphavantage.co/query'
        
        # Default high-volume tickers (you can modify this list)
        self.default_tickers = [
            'U', 'IONQ'
        ]
    
    def create_date_folder(self):
        """Create folder with MMDDYY format for today's date"""
        today = datetime.now()
        folder_name = today.strftime('%m%d%y')  # MMDDYY format
        
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
            print(f"📁 Created folder: {folder_name}")
        else:
            print(f"📁 Using existing folder: {folder_name}")
        
        return folder_name
    
    def get_daily_data(self, symbol):
        """Fetch daily data for a single ticker"""
        url = f'{self.base_url}?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=full&apikey={self.api_key}'
        
        try:
            print(f"Fetching data for {symbol}...", end=' ')
            response = requests.get(url)
            
            if response.status_code != 200:
                print(f"❌ HTTP Error: {response.status_code}")
                return None
            
            data = response.json()
            
            # Check for API errors
            if 'Error Message' in data:
                print(f"❌ API Error: {data['Error Message']}")
                return None
            
            if 'Note' in data:
                print(f"⚠️  API limit reached: {data['Note']}")
                return None
            
            if 'Time Series (Daily)' not in data:
                print(f"❌ No daily data available")
                return None
            
            # Extract time series data
            time_series = data['Time Series (Daily)']
            
            if not time_series:
                print(f"❌ Empty time series data")
                return None
            
            # Convert to DataFrame
            df = pd.DataFrame.from_dict(time_series, orient='index')
            
            # Rename columns to match standard format
            df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
            
            # Convert index to datetime and sort
            df.index = pd.to_datetime(df.index)
            df = df.sort_index()
            
            # Convert all columns to numeric
            df = df.apply(pd.to_numeric, errors='coerce')
            
            # Remove any rows with NaN values
            df = df.dropna()
            
            if len(df) == 0:
                print(f"❌ No valid data after cleaning")
                return None
            
            print(f"✅ Success - {len(df)} records from {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}")
            return df
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Network Error: {str(e)}")
            return None
        except Exception as e:
            print(f"❌ Unexpected Error: {str(e)}")
            return None
    
    def download_tickers(self, tickers=None, delay=12):
        """Download multiple tickers with rate limiting and save to date folder"""
        
        if tickers is None:
            tickers = self.default_tickers.copy()
        
        # Ensure tickers is a list
        if isinstance(tickers, str):
            tickers = [tickers]
        
        # Create date-based folder
        folder_name = self.create_date_folder()
        
        print("="*70)
        print("DAILY TICKER DATA DOWNLOADER")
        print("="*70)
        print(f"Date: {datetime.now().strftime('%B %d, %Y')}")
        print(f"Folder: {folder_name}")
        print(f"Downloading {len(tickers)} tickers")
        print(f"Rate limit: {delay} seconds between requests")
        print("-"*70)
        
        successful_downloads = []
        failed_downloads = []
        
        for i, ticker in enumerate(tickers):
            ticker = ticker.upper().strip()  # Normalize ticker format
            
            print(f"{i+1:2d}/{len(tickers)} {ticker}", end=" - ")
            
            # Download data
            df = self.get_daily_data(ticker)
            
            if df is not None:
                # Save to CSV with TICKERNAME_daily.csv format
                filename = f"{ticker}_daily.csv"
                filepath = os.path.join(folder_name, filename)
                
                try:
                    df.to_csv(filepath, index_label='Date')
                    
                    successful_downloads.append({
                        'ticker': ticker,
                        'filename': filename,
                        'filepath': filepath,
                        'records': len(df),
                        'date_range': f"{df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}",
                        'latest_close': df['Close'].iloc[-1]
                    })
                    print(f"   💾 Saved as {filename}")
                    
                except Exception as e:
                    print(f"   ❌ Save Error: {str(e)}")
                    failed_downloads.append(ticker)
            else:
                failed_downloads.append(ticker)
            
            # Rate limiting (Alpha Vantage free tier: 5 calls per minute)
            if i < len(tickers) - 1:  # Don't wait after the last request
                print(f"   ⏳ Waiting {delay} seconds...")
                time.sleep(delay)
        
        # Summary report
        self._print_summary(successful_downloads, failed_downloads, folder_name)
        
        return successful_downloads, failed_downloads, folder_name
    
    def _print_summary(self, successful_downloads, failed_downloads, folder_name):
        """Print download summary report"""
        print("\n" + "="*70)
        print("DOWNLOAD SUMMARY")
        print("="*70)
        
        if successful_downloads:
            print(f"✅ Successfully downloaded {len(successful_downloads)} tickers:")
            print("-"*70)
            print(f"{'Ticker':<8} {'Records':<8} {'Date Range':<25} {'Latest Close':<12}")
            print("-"*70)
            for item in successful_downloads:
                print(f"{item['ticker']:<8} {item['records']:<8} {item['date_range']:<25} ${item['latest_close']:<11.2f}")
        
        if failed_downloads:
            print(f"\n❌ Failed downloads ({len(failed_downloads)}):")
            print(", ".join(failed_downloads))
        
        print(f"\n📁 Files saved to folder: {os.path.abspath(folder_name)}/")
        print(f"📊 Total files: {len(successful_downloads)}")
        
        if successful_downloads:
            print(f"\n🎯 Files ready for analysis:")
            for item in successful_downloads[:5]:  # Show first 5 as examples
                print(f"   {item['filepath']}")
            if len(successful_downloads) > 5:
                print(f"   ... and {len(successful_downloads) - 5} more files")
    
    def download_single_ticker(self, ticker, delay=0):
        """Download a single ticker"""
        return self.download_tickers([ticker], delay)
    
    def download_custom_list(self, ticker_list, delay=12):
        """Download a custom list of tickers"""
        return self.download_tickers(ticker_list, delay)


def main():
    """Main function with user interface"""
    # Your Alpha Vantage API key
    API_KEY = 'OIYK9UHAU6E7H57X'
    
    downloader = DailyTickerDownloader(API_KEY)
    
    print("DAILY TICKER DATA DOWNLOADER")
    print("="*50)
    print(f"Today's folder will be: {datetime.now().strftime('%m%d%y')}")
    print()
    
    # Menu options
    options = {
        '1': 'Download default ticker list (27 tickers)',
        '2': 'Download single ticker',
        '3': 'Download custom ticker list',
        '4': 'Download top 10 liquid tickers only'
    }
    
    print("Choose download option:")
    for key, description in options.items():
        print(f"{key}. {description}")
    
    choice = input("\nEnter choice (1-4): ").strip()
    
    try:
        if choice == '1':
            print("\nDownloading default ticker list...")
            successful, failed, folder = downloader.download_tickers()
            
        elif choice == '2':
            ticker = input("Enter ticker symbol: ").strip().upper()
            if ticker:
                print(f"\nDownloading {ticker}...")
                successful, failed, folder = downloader.download_single_ticker(ticker)
            else:
                print("❌ Invalid ticker symbol")
                return
                
        elif choice == '3':
            ticker_input = input("Enter ticker symbols (comma-separated): ").strip()
            if ticker_input:
                ticker_list = [t.strip().upper() for t in ticker_input.split(',')]
                print(f"\nDownloading {len(ticker_list)} tickers: {', '.join(ticker_list)}")
                successful, failed, folder = downloader.download_custom_list(ticker_list)
            else:
                print("❌ No tickers provided")
                return
                
        elif choice == '4':
            top_10 = ['PLTR', 'COIN', 'ENPH', 'RBLX', 'DKNG', 'IONQ', 'HOOD', 'AMD', 'MU', 'AVGO', 'ARM', 'U']
            print(f"\nDownloading top 10 liquid tickers: {', '.join(top_10)}")
            successful, failed, folder = downloader.download_custom_list(top_10)
            
        else:
            print("❌ Invalid choice. Exiting...")
            return
        
        print(f"\n✅ Download complete!")
        print(f"📁 Check folder '{folder}' for your CSV files")
        
    except KeyboardInterrupt:
        print("\n\n⚠️  Download interrupted by user")
    except Exception as e:
        print(f"\n❌ Unexpected error: {str(e)}")


if __name__ == "__main__":
    main()

DAILY TICKER DATA DOWNLOADER
Today's folder will be: 111025

Choose download option:
1. Download default ticker list (27 tickers)
2. Download single ticker
3. Download custom ticker list
4. Download top 10 liquid tickers only

📁 Created folder: 111025
DAILY TICKER DATA DOWNLOADER
Date: November 10, 2025
Folder: 111025
Rate limit: 12 seconds between requests
----------------------------------------------------------------------
 1/11 COIN - Fetching data for COIN... ✅ Success - 1151 records from 2021-04-14 to 2025-11-10
   💾 Saved as COIN_daily.csv
   ⏳ Waiting 12 seconds...
 2/11 ENPH - Fetching data for ENPH... ✅ Success - 3424 records from 2012-03-30 to 2025-11-10
   💾 Saved as ENPH_daily.csv
   ⏳ Waiting 12 seconds...
 3/11 RBLX - Fetching data for RBLX... ✅ Success - 1175 records from 2021-03-10 to 2025-11-10
   💾 Saved as RBLX_daily.csv
   ⏳ Waiting 12 seconds...
 4/11 DKNG - Fetching data for DKNG... ✅ Success - 1583 records from 2019-07-25 to 2025-11-10
   💾 Saved as DKNG_daily.